In [29]:
#set up
# import libraries
!pip install numpy pandas matplotlib seaborn yfinance datetime pandas_datareader statsmodels
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns
import yfinance as yf
import datetime
from datetime import datetime, timedelta


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [30]:
TICKER = 'BRK-B'
START  = '1980-01-01'
END    = datetime.today().strftime('%Y-%m-%d')

# Download daily prices and resample to true month-end closes
brk_daily   = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
brk_monthly = brk_daily['Close'].resample('ME').last()

# .squeeze() converts the single-column DataFrame yfinance returns into a Series
brk_ret       = brk_monthly.pct_change().dropna().squeeze()
brk_ret.index = brk_ret.index.to_period('M')
brk_ret.name  = 'BRK_ret'

print(f'BRK-B monthly returns: {brk_ret.index[0]} – {brk_ret.index[-1]}  ({len(brk_ret)} months)')
brk_ret.head(10)

BRK-B monthly returns: 1996-06 – 2026-03  (358 months)


Date
1996-06    0.012745
1996-07   -0.006776
1996-08    0.014620
1996-09    0.030740
1996-10    0.005592
1996-11    0.022243
1996-12    0.008160
1997-01    0.038669
1997-02    0.028571
1997-03    0.018519
Freq: M, Name: BRK_ret, dtype: float64

In [31]:
# Fama-French 5 factors — downloaded directly from Ken French's data library
import io, zipfile, requests

url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip'
r   = requests.get(url)
z   = zipfile.ZipFile(io.BytesIO(r.content))
csv = z.read(z.namelist()[0]).decode('utf-8')

# Skip the header description rows; stop before the annual block (first blank line after monthly data)
lines       = csv.splitlines()
data_start  = next(i for i, l in enumerate(lines) if l.strip().startswith('196') or l.strip().startswith('192'))
data_lines  = []
for line in lines[data_start:]:
    if line.strip() == '':
        break
    data_lines.append(line)

ff5 = pd.read_csv(io.StringIO('\n'.join(data_lines)), header=None,
                  names=['Date','Mkt-RF','SMB','HML','RMW','CMA','RF'])
ff5['Date'] = pd.to_datetime(ff5['Date'].astype(str).str.strip(), format='%Y%m')
ff5 = ff5.set_index('Date')
ff5 = ff5.apply(pd.to_numeric) / 100
ff5.index = ff5.index.to_period('M')
ff5 = ff5[ff5.index >= pd.Period(START, 'M')]

print(f'FF5 factors: {ff5.index[0]} – {ff5.index[-1]}  ({len(ff5)} months)')
ff5.head()

FF5 factors: 1980-01 – 2026-02  (554 months)


,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
1980-01,0.0550,0.0188,0.0185,-0.0184,0.0189,0.0080
1980-02,-0.0123,-0.0162,0.0059,-0.0095,0.0292,0.0089
1980-03,-0.1289,-0.0697,-0.0096,0.0182,-0.0105,0.0121
1980-04,0.0396,0.0105,0.0103,-0.0218,0.0034,0.0126
1980-05,0.0526,0.0200,0.0038,0.0043,-0.0063,0.0081


In [32]:
# Merge FF5 factors with BRK-B returns and compute excess return
df = ff5.join(brk_ret, how='inner')
df['BRK_excess'] = df['BRK_ret'] - df['RF']
df = df[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'BRK_ret', 'BRK_excess']]

print(f'Merged DataFrame: {df.index[0]} – {df.index[-1]}  ({len(df)} months)')
df.head(10)


Merged DataFrame: 1996-06 – 2026-02  (357 months)


,Mkt-RF,SMB,HML,RMW,CMA,RF,BRK_ret,BRK_excess
Date,,,,,,,,
1996-06,-0.0113,-0.0367,0.0233,0.0350,0.0120,0.0040,0.012745,0.008745
1996-07,-0.0596,-0.0378,0.0519,0.0290,0.0268,0.0045,-0.006776,-0.011276
1996-08,0.0277,0.0258,-0.0082,-0.0038,-0.0247,0.0041,0.014620,0.010520
1996-09,0.0501,-0.0141,-0.0274,0.0130,-0.0213,0.0044,0.030740,0.026340
1996-10,0.0087,-0.0379,0.0488,0.0138,0.0297,0.0042,0.005592,0.001392
1996-11,0.0625,-0.0382,0.0140,0.0206,-0.0069,0.0041,0.022243,0.018143
1996-12,-0.0170,0.0319,0.0128,0.0023,0.0149,0.0046,0.008160,0.003560
1997-01,0.0497,-0.0182,-0.0144,0.0122,-0.0007,0.0045,0.038669,0.034169
1997-02,-0.0048,-0.0254,0.0561,0.0072,0.0347,0.0039,0.028571,0.024671


In [33]:
import statsmodels.api as sm

y = df['BRK_excess']

# CAPM (1-factor)
X_capm = sm.add_constant(df[['Mkt-RF']])
capm   = sm.OLS(y, X_capm).fit()

# Fama-French 3-factor
X_ff3 = sm.add_constant(df[['Mkt-RF', 'SMB', 'HML']])
ff3   = sm.OLS(y, X_ff3).fit()

# Fama-French 5-factor
X_ff5 = sm.add_constant(df[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']])
ff5r  = sm.OLS(y, X_ff5).fit()

# Summary table
results = pd.DataFrame({
    'CAPM':    [capm.params['const'],  capm.pvalues['const'],  capm.rsquared],
    'FF3':     [ff3.params['const'],   ff3.pvalues['const'],   ff3.rsquared],
    'FF5':     [ff5r.params['const'],  ff5r.pvalues['const'],  ff5r.rsquared],
}, index=['Alpha (monthly)', 'Alpha p-value', 'R²'])

# Annualise alpha for readability
results.loc['Alpha (annualised)'] = results.loc['Alpha (monthly)'] * 12

print(results.round(4))

                      CAPM     FF3     FF5
Alpha (monthly)     0.0045  0.0030  0.0021
Alpha p-value       0.0832  0.1883  0.3581
R²                  0.2283  0.4049  0.4131
Alpha (annualised)  0.0538  0.0360  0.0257


In [34]:
# Full statsmodels output for FF5 (most complete model)
print(ff5r.summary())

                            OLS Regression Results                            
Dep. Variable:             BRK_excess   R-squared:                       0.413
Model:                            OLS   Adj. R-squared:                  0.405
Method:                 Least Squares   F-statistic:                     49.41
Date:                Tue, 31 Mar 2026   Prob (F-statistic):           1.15e-38
Time:                        12:52:22   Log-Likelihood:                 626.15
No. Observations:                 357   AIC:                            -1240.
Df Residuals:                     351   BIC:                            -1217.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0021      0.002      0.920      0.3